In [1]:
import joblib

import numpy as np
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer ,PorterStemmer

from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer

from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB  
from hmmlearn.hmm import GaussianHMM

# from sktime.detection.hmm_learn import GaussianHMM 

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.decomposition import TruncatedSVD #instead of PCA
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler


In [2]:
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\20100\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\20100\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\20100\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\20100\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
stop_words=set(stopwords.words('english'))

# Load the data + class Weights

In [4]:
data = joblib.load('news_data.pkl')
x_train = data['X_train']
y_train = data['y_train']
x_test = data['X_test']
y_test = data['y_test']

data_balance = joblib.load('news_data_resampled.pkl')
x_train_ran_res = data_balance['X_train']
y_train_ran_res = data_balance['y_train']
x_test_ran_res = data_balance['X_test']
y_test_ran_res = data_balance['y_test']

data_balance_rosrus=joblib.load('news_data_bal_ros_rus.pkl') #done
x_train_bal = data_balance_rosrus['X_train']
y_train_bal = data_balance_rosrus['y_train']
x_test_bal = data_balance_rosrus['X_test']
y_test_bal = data_balance_rosrus['y_test']

In [5]:
data_undersampled=joblib.load('news_data_undersampled.pkl') # in progress
x_train_undersampled=data_undersampled['X_train']
y_train_undersampled=data_undersampled['y_train']
x_test_undersampled=data_undersampled['X_test']
y_test_undersampled=data_undersampled['y_test']

In [6]:
class_weights_dict=joblib.load('classWeightsDic')

# Initial  POS

In [7]:
def pos_features(text, remove_stopwords=False):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    tokens = word_tokenize(text.lower())

    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words]

    pos_tags = nltk.pos_tag(tokens)
    pos_tokens = [tag for _, tag in pos_tags]
    return pos_tokens


In [8]:
num_classes = len(set(y_train))  
num_classes

10

# Original Data

In [9]:
len(x_train)

167616

## BoW feature Extraction + scalling

## count vectorizer

In [10]:
vectorizer_POS_ = CountVectorizer(tokenizer=lambda x:pos_features(x, remove_stopwords=True))
x_train_POS = vectorizer_POS_.fit_transform(x_train)
x_test_POS= vectorizer_POS_.transform(x_test)

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [11]:
print(x_train_POS)

  (0, 11)	3
  (0, 14)	2
  (0, 27)	1
  (0, 7)	1
  (1, 11)	2
  (1, 14)	1
  (1, 30)	1
  (1, 6)	1
  (1, 29)	2
  (1, 18)	1
  (2, 11)	5
  (2, 7)	2
  (2, 30)	1
  (3, 11)	2
  (3, 7)	2
  (3, 26)	1
  (4, 11)	2
  (4, 14)	3
  (4, 7)	2
  (4, 29)	1
  (4, 26)	1
  (5, 11)	5
  (5, 27)	1
  (5, 7)	1
  (5, 30)	1
  :	:
  (167610, 14)	1
  (167610, 7)	1
  (167610, 29)	1
  (167610, 26)	1
  (167611, 11)	5
  (167611, 14)	1
  (167611, 27)	1
  (167611, 7)	1
  (167611, 29)	1
  (167612, 11)	2
  (167612, 14)	1
  (167612, 7)	2
  (167613, 11)	3
  (167613, 14)	1
  (167613, 7)	2
  (167613, 29)	1
  (167613, 18)	1
  (167613, 26)	1
  (167614, 11)	3
  (167614, 7)	2
  (167614, 18)	1
  (167615, 11)	6
  (167615, 14)	1
  (167615, 7)	1
  (167615, 29)	1


In [12]:
pos_counts = np.asarray(x_train_POS.sum(axis=0)).ravel()
pos_names = vectorizer_POS_.get_feature_names_out()

pos_freq = pd.Series(pos_counts, index=pos_names).sort_values(ascending=False)
pos_freq

NN      463065
NNS     179788
JJ      178187
VBP      66311
VBG      43699
VBZ      34250
RB       33917
VBD      28797
VB       20555
VBN      12995
IN       11031
JJS       6877
CD        6495
MD        6150
PRP       4135
JJR       2808
RBR       1731
DT        1728
RP        1010
FW         878
CC         656
NNP        650
RBS        446
WRB        225
WP         114
WP$         97
NNPS        79
UH          71
WDT         57
PRP$        49
TO          39
POS          9
''           8
SYM          5
EX           5
dtype: int64

In [13]:
scaler = StandardScaler(with_mean=False)
x_train_pos_scaled = scaler.fit_transform(x_train_POS)
x_test_pos_scaled  = scaler.transform(x_test_POS)

In [14]:
vectorizer_POS_stopKept = CountVectorizer(tokenizer=lambda x: pos_features(x, remove_stopwords=False))
x_train_POS_stopkept = vectorizer_POS_stopKept.fit_transform(x_train)
x_test_POS_stopkept = vectorizer_POS_stopKept.transform(x_test)

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [15]:
scaler_stopkept = StandardScaler(with_mean=False)
x_train_pos_stopkept_scaled = scaler_stopkept.fit_transform(x_train_POS_stopkept)
x_test_pos_stopkept_scaled  = scaler_stopkept.transform(x_test_POS_stopkept)

## TF-IDF vectorizer

In [16]:
vectorizer_POS_tfidf = TfidfVectorizer(tokenizer=lambda x: pos_features(x, remove_stopwords=True))
x_train_POS_tfidf = vectorizer_POS_tfidf.fit_transform(x_train)
x_test_POS_tfidf = vectorizer_POS_tfidf.transform(x_test)

In [17]:
scaler_tfidf = StandardScaler(with_mean=False)
x_train_pos_tfidf_scaled = scaler_tfidf.fit_transform(x_train_POS_tfidf)
x_test_pos_tfidf_scaled  = scaler_tfidf.transform(x_test_POS_tfidf)

In [18]:
vectorizer_POS_stopkept_tfidf = TfidfVectorizer(tokenizer=lambda x: pos_features(x, remove_stopwords=True))
x_train_POS_stopkept_tfidf = vectorizer_POS_stopkept_tfidf.fit_transform(x_train)
x_test_POS_stopkept_tfidf = vectorizer_POS_stopkept_tfidf.transform(x_test)

In [19]:
scaler_stopkept_tfidf = StandardScaler(with_mean=False)
x_train_pos_stopkept_tfidf_scaled = scaler_stopkept_tfidf.fit_transform(x_train_POS_stopkept_tfidf)
x_test_pos_stopkept_tfidf_scaled  = scaler_stopkept_tfidf.transform(x_test_POS_stopkept_tfidf)

## Models

In [52]:
# results_POS_original={}
joblib.dump(results_POS_original,'results_POS_original')

['results_POS_original']

In [53]:
results_POS_original=joblib.load('results_POS_original')
results_POS_original

{'SVC_BoW StopWords removed': 0.13430378236487292,
 'MultinomialNB_BoW StopWords removed': 0.40746927574275144,
 'MLP StopWords removed': 0.4539076482519986,
 'SVC_POS StopWords kept': 0.16735473093902875,
 'MultinomialNB_POS StopWords kept': 0.37441832716859563,
 'MLP StopWords kept': 0.45825080539315116,
 '(tf-idf) SVC_POS StopWords removed': 0.12208566996778428,
 '(tf-idf) MultinomialNB_POS StopWords removed': 0.39661138288986997,
 '(tf-idf) MLP StopWords removed': 0.45426560076363204,
 '(tf-idf) SVC_POS StopWords kept': 0.12208566996778428,
 '(tf-idf) MultinomialNB_POS StopWords kept': 0.39661138288986997,
 '(tf-idf) MLP StopWords kept': 0.45426560076363204}

In [22]:
x_train_pos_tfidf_scaled.shape

(167616, 35)

In [23]:
x_train_pos_scaled.shape

(167616, 35)

In [40]:
models_POS={
    "SVC_POS": SVC(class_weight=class_weights_dict),
    "MultinomialNB_POS": MultinomialNB(),
    "MLP" :MLPClassifier(hidden_layer_sizes=(32,16),activation='relu',solver='adam',max_iter=100,alpha=0.001,early_stopping=True,random_state=42),
    # 'HMM' :GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
    }

## with countvectorizer

### stop words removed

In [25]:
for model_name, model in models_POS.items():
    model.fit(x_train_pos_scaled, y_train)

    y_pred_train = model.predict(x_train_pos_scaled)
    accuracy_train=accuracy_score(y_train, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_pos_scaled)
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name}Testing: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)
    results_POS_original[model_name+" StopWords removed"] = accuracy

Results for SVC_BoW Training: accuracy=0.14853593928980527
Results for SVC_BoWTesting: accuracy=0.13430378236487292
              precision    recall  f1-score   support

           1       0.24      0.17      0.20      7120
           2       0.20      0.34      0.25      3589
           3       0.15      0.26      0.19      3473
           4       0.12      0.24      0.16      1980
           5       0.11      0.28      0.15      1963
           6       0.04      0.08      0.05      1269
           7       0.06      0.34      0.10      1268
           8       0.04      0.09      0.06      1198
           9       0.05      0.12      0.07      1016
          10       0.58      0.03      0.05     19029

    accuracy                           0.13     41905
   macro avg       0.16      0.19      0.13     41905
weighted avg       0.35      0.13      0.12     41905

--------------------------------------------------
Results for MultinomialNB_BoW Training: accuracy=0.4088989117983963
Result

d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [41]:
results_POS_original

{'SVC_BoW StopWords removed': 0.13430378236487292,
 'MultinomialNB_BoW StopWords removed': 0.40746927574275144,
 'MLP StopWords removed': 0.4539076482519986}

#### HMM 

In [33]:
svd_POS = TruncatedSVD(n_components=20, random_state=42)
x_train_svd_POS = svd_POS.fit_transform(x_train_POS)
x_test_svd_POS = svd_POS.transform(x_test_POS)

scaler = StandardScaler()
x_train_svd_POS = scaler.fit_transform(x_train_svd_POS)
x_test_svd_POS = scaler.transform(x_test_svd_POS)

In [29]:
hmm_POS=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

In [35]:
lengths_train = [1] * x_train_svd_POS.shape[0]
lengths_test = [1] * x_test_svd_POS.shape[0]

In [34]:
hmm_POS.fit(x_train_svd_POS)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [39]:
y_pred_POS_original=hmm_POS.predict(x_train_svd_POS,)
accuracy_hmm_POS_original = accuracy_score(y_train, y_pred_POS_original)   
print(f"training HMM POS Accuracy : {accuracy_hmm_POS_original}")

training HMM POS Accuracy : 0.0362077605956472


In [37]:
y_pred_POS_original=hmm_POS.predict(x_test_svd_POS,lengths=lengths_test)
accuracy_hmm_POS_original = accuracy_score(y_test, y_pred_POS_original)   
print(f"testing HMM POS Accuracy with length : {accuracy_hmm_POS_original}")

testing HMM POS Accuracy with length : 0.022551008232907767


In [38]:
y_pred_POS_original=hmm_POS.predict(x_test_svd_POS,)
accuracy_hmm_POS_original = accuracy_score(y_test, y_pred_POS_original)   
print(f"testing HMM POS Accuracy : {accuracy_hmm_POS_original}")

testing HMM POS Accuracy : 0.03667820069204152


In [80]:
results_POS_original['HMM StopWords removed']=0.03667820069204152

### stopwords kept

In [42]:
for model_name, model in models_POS.items():
    model.fit(x_train_pos_stopkept_scaled, y_train)

    y_pred_train = model.predict(x_train_pos_stopkept_scaled)
    accuracy_train=accuracy_score(y_train, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_pos_stopkept_scaled)
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)
    results_POS_original[model_name+" StopWords kept"] = accuracy

Results for SVC_POS Training: accuracy=0.1991635643375334
Results for SVC_POS Testing: accuracy=0.16735473093902875
              precision    recall  f1-score   support

           1       0.32      0.21      0.25      7120
           2       0.22      0.37      0.28      3589
           3       0.18      0.29      0.22      3473
           4       0.13      0.34      0.19      1980
           5       0.17      0.30      0.22      1963
           6       0.06      0.14      0.09      1269
           7       0.10      0.35      0.16      1268
           8       0.06      0.12      0.08      1198
           9       0.04      0.24      0.07      1016
          10       0.60      0.05      0.09     19029

    accuracy                           0.17     41905
   macro avg       0.19      0.24      0.16     41905
weighted avg       0.38      0.17      0.16     41905

--------------------------------------------------
Results for MultinomialNB_POS Training: accuracy=0.37417072355861014
Resul

d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#### HMM

In [ ]:
hmm_POS_stopkept=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

In [55]:
svd_POS_stopkept = TruncatedSVD(n_components=35, random_state=42)
x_train_svd_POS_stopkept = svd_POS_stopkept.fit_transform(x_train_POS_stopkept)
x_test_svd_POS_stopkept = svd_POS_stopkept.transform(x_test_POS_stopkept)

scaler_stopkept = StandardScaler()
x_train_svd_POS_stopkept_scaled = scaler_stopkept.fit_transform(x_train_svd_POS_stopkept)
x_test_svd_POS_stopkept_scaled = scaler_stopkept.transform(x_test_svd_POS_stopkept)

In [56]:
lengths_train = [1] * x_train_svd_POS_stopkept_scaled.shape[0]
lengths_test = [1] * x_test_svd_POS_stopkept_scaled.shape[0]

In [57]:
hmm_POS_stopkept.fit(x_train_svd_POS_stopkept_scaled)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [59]:
y_pred_POS_stopkept_original=hmm_POS_stopkept.predict(x_train_svd_POS_stopkept_scaled,)
accuracy_hmm_POS_stopkept_original = accuracy_score(y_train, y_pred_POS_stopkept_original)   
print(f"training HMM POS Accuracy : {accuracy_hmm_POS_stopkept_original}")

training HMM POS Accuracy : 0.05114666857579229


In [60]:
y_pred_POS_stopkept_original=hmm_POS_stopkept.predict(x_test_svd_POS_stopkept_scaled,lengths=lengths_test)
accuracy_hmm_POS_stopkept_original = accuracy_score(y_test, y_pred_POS_stopkept_original)   
print(f"testing HMM POS Accuracy : {accuracy_hmm_POS_stopkept_original}")

testing HMM POS Accuracy : 0.0


In [61]:
y_pred_POS_stopkept_original=hmm_POS_stopkept.predict(x_test_svd_POS_stopkept_scaled)
accuracy_hmm_POS_stopkept_original = accuracy_score(y_test, y_pred_POS_stopkept_original)   
print(f"testing HMM POS Accuracy : {accuracy_hmm_POS_stopkept_original}")

testing HMM POS Accuracy : 0.0530962892256294


In [81]:
results_POS_original['HMM StopWords kept']=0.0530962892256294

## With tf-idf

### stopwords removed

In [43]:
for model_name, model in models_POS.items():
    model.fit(x_train_pos_tfidf_scaled, y_train)

    y_pred_train = model.predict(x_train_pos_tfidf_scaled)
    accuracy_train=accuracy_score(y_train, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_pos_tfidf_scaled)
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)
    results_POS_original["(tf-idf) "+model_name+" StopWords removed"] = accuracy

Results for SVC_POS Training: accuracy=0.1330541237113402
Results for SVC_POS Testing: accuracy=0.12208566996778428
              precision    recall  f1-score   support

           1       0.25      0.11      0.15      7120
           2       0.17      0.35      0.23      3589
           3       0.14      0.24      0.17      3473
           4       0.10      0.29      0.14      1980
           5       0.09      0.27      0.14      1963
           6       0.04      0.05      0.04      1269
           7       0.06      0.18      0.09      1268
           8       0.04      0.10      0.06      1198
           9       0.03      0.12      0.05      1016
          10       0.56      0.03      0.06     19029

    accuracy                           0.12     41905
   macro avg       0.15      0.18      0.11     41905
weighted avg       0.34      0.12      0.11     41905

--------------------------------------------------
Results for MultinomialNB_POS Training: accuracy=0.39814218213058417
Resul

d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#### HMM

In [74]:
svd_POS_tfidf = TruncatedSVD(n_components=20, random_state=42)
x_train_svd_POS_tfidf = svd_POS_tfidf.fit_transform(x_train_POS_tfidf)
x_test_svd_POS_tfidf = svd_POS_tfidf.transform(x_test_POS_tfidf)

scaler = StandardScaler()
x_train_svd_POS_tfidf_scaled = scaler.fit_transform(x_train_svd_POS_tfidf)
x_test_svd_POS_tfidf_scaled = scaler.transform(x_test_svd_POS_tfidf)

In [72]:
hmm_POS_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

In [75]:
lengths_train = [1] * x_train_svd_POS_tfidf_scaled.shape[0]
lengths_test = [1] * x_test_svd_POS_tfidf_scaled.shape[0]

In [76]:
hmm_POS_tfidf.fit(x_train_svd_POS_tfidf_scaled)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [77]:
y_pred_POS_original_tfidf=hmm_POS_tfidf.predict(x_train_svd_POS_tfidf_scaled,lengths=lengths_train)
accuracy_hmm_POS_original = accuracy_score(y_train, y_pred_POS_original_tfidf)   
print(f"training HMM POS Accuracy : {accuracy_hmm_POS_original}")

training HMM POS Accuracy : 0.04725085910652921


In [78]:
y_pred_POS_original_tfidf=hmm_POS_tfidf.predict(x_test_svd_POS_tfidf_scaled,lengths=lengths_test)
accuracy_hmm_POS_original = accuracy_score(y_test, y_pred_POS_original_tfidf)   
print(f"testing HMM POS Accuracy : {accuracy_hmm_POS_original}")

testing HMM POS Accuracy : 0.04724973153561628


In [79]:
y_pred_POS_original_tfidf=hmm_POS_tfidf.predict(x_test_svd_POS_tfidf_scaled,)
accuracy_hmm_POS_original = accuracy_score(y_test, y_pred_POS_original_tfidf)   
print(f"testing HMM POS Accuracy : {accuracy_hmm_POS_original}")

testing HMM POS Accuracy : 0.039995227299844886


In [82]:
results_POS_original['(tf-idf) HMM StopWords removed']=0.04724973153561628

### stopwords kept

In [ ]:
for model_name, model in models_POS.items():
    model.fit(x_train_pos_stopkept_tfidf_scaled, y_train)

    y_pred_train = model.predict(x_train_pos_stopkept_tfidf_scaled)
    accuracy_train=accuracy_score(y_train, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_pos_stopkept_tfidf_scaled)
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)
    results_POS_original['(tf-idf) '+model_name+" StopWords kept"] = accuracy

Results for SVC_POS Training: accuracy=0.1330541237113402
Results for SVC_POS Testing: accuracy=0.12208566996778428
              precision    recall  f1-score   support

           1       0.25      0.11      0.15      7120
           2       0.17      0.35      0.23      3589
           3       0.14      0.24      0.17      3473
           4       0.10      0.29      0.14      1980
           5       0.09      0.27      0.14      1963
           6       0.04      0.05      0.04      1269
           7       0.06      0.18      0.09      1268
           8       0.04      0.10      0.06      1198
           9       0.03      0.12      0.05      1016
          10       0.56      0.03      0.06     19029

    accuracy                           0.12     41905
   macro avg       0.15      0.18      0.11     41905
weighted avg       0.34      0.12      0.11     41905

--------------------------------------------------
Results for MultinomialNB_POS Training: accuracy=0.39814218213058417
Resul

d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#### HMM

In [62]:
hmm_POS_stopkept_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

In [63]:
svd_POS_stopkept_tfidf = TruncatedSVD(n_components=35, random_state=42)
x_train_svd_POS_stopkept_tfidf = svd_POS_stopkept_tfidf.fit_transform(x_train_POS_stopkept_tfidf)
x_test_svd_POS_stopkept_tfidf = svd_POS_stopkept_tfidf.transform(x_test_POS_stopkept_tfidf)

scaler_stopkept_tfidf = StandardScaler()
x_train_svd_POS_stopkept_tfidf_scaled = scaler_stopkept_tfidf.fit_transform(x_train_svd_POS_stopkept_tfidf)
x_test_svd_POS_stopkept_tfidf_scaled = scaler_stopkept_tfidf.transform(x_test_svd_POS_stopkept_tfidf)

In [64]:
lengths_train = [1] * x_train_svd_POS_stopkept_tfidf_scaled.shape[0]
lengths_test = [1] * x_test_svd_POS_stopkept_tfidf_scaled.shape[0]

In [65]:
hmm_POS_stopkept_tfidf.fit(x_train_svd_POS_stopkept_tfidf_scaled)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [70]:
y_pred_POS_stopkept_tfidf_original=hmm_POS_stopkept_tfidf.predict(x_train_svd_POS_stopkept_tfidf_scaled,lengths=lengths_train)
accuracy_hmm_POS_stopkept_tfidf_original = accuracy_score(y_train, y_pred_POS_stopkept_tfidf_original)   
print(f"training HMM POS Accuracy : {accuracy_hmm_POS_stopkept_tfidf_original}")

training HMM POS Accuracy : 0.08564814814814815


In [67]:
y_pred_POS_stopkept_tfidf_original=hmm_POS_stopkept_tfidf.predict(x_test_svd_POS_stopkept_tfidf_scaled,)
accuracy_hmm_POS_stopkept_tfidf_original = accuracy_score(y_test, y_pred_POS_stopkept_tfidf_original)   
print(f"testing HMM POS Accuracy : {accuracy_hmm_POS_stopkept_tfidf_original}")

testing HMM POS Accuracy : 0.06545758262737143


In [69]:
y_pred_POS_stopkept_tfidf_original=hmm_POS_stopkept_tfidf.predict(x_test_svd_POS_stopkept_tfidf_scaled,lengths=lengths_test)
accuracy_hmm_POS_stopkept_tfidf_original = accuracy_score(y_test, y_pred_POS_stopkept_tfidf_original)   
print(f"testing HMM POS Accuracy : {accuracy_hmm_POS_stopkept_tfidf_original}")

testing HMM POS Accuracy : 0.08564610428349839


In [83]:
results_POS_original['(tf-idf) HMM StopWords kept']=0.08564610428349839

## Save results

In [84]:
results_POS_original

{'SVC_BoW StopWords removed': 0.13430378236487292,
 'MultinomialNB_BoW StopWords removed': 0.40746927574275144,
 'MLP StopWords removed': 0.4539076482519986,
 'SVC_POS StopWords kept': 0.16735473093902875,
 'MultinomialNB_POS StopWords kept': 0.37441832716859563,
 'MLP StopWords kept': 0.45825080539315116,
 '(tf-idf) SVC_POS StopWords removed': 0.12208566996778428,
 '(tf-idf) MultinomialNB_POS StopWords removed': 0.39661138288986997,
 '(tf-idf) MLP StopWords removed': 0.45426560076363204,
 '(tf-idf) SVC_POS StopWords kept': 0.12208566996778428,
 '(tf-idf) MultinomialNB_POS StopWords kept': 0.39661138288986997,
 '(tf-idf) MLP StopWords kept': 0.45426560076363204,
 'HMM StopWords removed': 0.03667820069204152,
 'HMM StopWords kept': 0.0530962892256294,
 '(tf-idf) HMM StopWords removed': 0.04724973153561628,
 '(tf-idf) HMM StopWords kept': 0.08564610428349839}

In [85]:
joblib.dump(results_POS_original,'results_POS_original')

['results_POS_original']

In [86]:
results_POS_original=joblib.load('results_POS_original')
results_POS_original

{'SVC_BoW StopWords removed': 0.13430378236487292,
 'MultinomialNB_BoW StopWords removed': 0.40746927574275144,
 'MLP StopWords removed': 0.4539076482519986,
 'SVC_POS StopWords kept': 0.16735473093902875,
 'MultinomialNB_POS StopWords kept': 0.37441832716859563,
 'MLP StopWords kept': 0.45825080539315116,
 '(tf-idf) SVC_POS StopWords removed': 0.12208566996778428,
 '(tf-idf) MultinomialNB_POS StopWords removed': 0.39661138288986997,
 '(tf-idf) MLP StopWords removed': 0.45426560076363204,
 '(tf-idf) SVC_POS StopWords kept': 0.12208566996778428,
 '(tf-idf) MultinomialNB_POS StopWords kept': 0.39661138288986997,
 '(tf-idf) MLP StopWords kept': 0.45426560076363204,
 'HMM StopWords removed': 0.03667820069204152,
 'HMM StopWords kept': 0.0530962892256294,
 '(tf-idf) HMM StopWords removed': 0.04724973153561628,
 '(tf-idf) HMM StopWords kept': 0.08564610428349839}

In [87]:
with open("results_POS_original.txt", "w", encoding="utf-8") as f:
    for key, value in results_POS_original.items():
        f.write(f"{key}: {value}\n")

# ______________________________________________________________________________________

# resampled Data (undersample)

In [12]:
# results_Bow_undersample={}
results_Bow_undersample=joblib.load('results_Bow_undersample')
results_Bow_undersample

{'undersample SVM stem+StopWords removed': 0.6263328141847371,
 'undersample MultinomialNB stem+StopWords removed': 0.6012938780400143,
 'undersample MLP stem+StopWords removed': 0.6314843656403498,
 'undersample SVM stem+StopWords kept': 0.6225590032346952,
 'undersample MultinomialNB stem+StopWords kept': 0.5940457649454894,
 'undersample MLP stem+StopWords kept': 0.6299868216125554,
 'undersample SVM lemma+StopWords removed': 0.6224391997124715,
 'undersample MultinomialNB lemma+StopWords removed': 0.5948843896010543,
 'undersample MLP lemma+StopWords removed': 0.6245357613513838,
 'undersample SVM lemma+StopWords kept': 0.6184856834790943,
 'undersample MultinomialNB lemma+StopWords kept': 0.5880555888343117,
 'undersample MLP lemma+StopWords kept': 0.6229783155624775,
 'undersample HMM stem+StopWords removed': 0.0770935665508566,
 'undersample HMM stem+StopWords kept': 0.11980352222355337,
 'undersample HMM lemma+StopWords removed': 0.08649814304540554,
 'undersample HMM lemma+Sto

## BoW feature Extraction

### with countvectorizer

In [30]:
vectorizer_stem_undersample=CountVectorizer(tokenizer=lambda x : stem(x,remove_stopwords=True),max_features=1000)
x_train_stem_undersampled=vectorizer_stem_undersample.fit_transform(x_train_undersampled)
x_test_stem_undersampled=vectorizer_stem_undersample.transform(x_test_undersampled)

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [14]:
vectorizer_stem_stopkept_undersample=CountVectorizer(tokenizer=lambda x : stem(x,remove_stopwords=False),max_features=1000)
x_train_stem_stopkept_undersampled=vectorizer_stem_stopkept_undersample.fit_transform(x_train_undersampled)
x_test_stem_stopkept_undersampled=vectorizer_stem_stopkept_undersample.transform(x_test_undersampled)

In [15]:
vectorizer_lemma_undersample=CountVectorizer(tokenizer=lambda x : lemma(x,remove_stopwords=True),max_features=1000)
x_train_lemma_undersampled=vectorizer_lemma_undersample.fit_transform(x_train_undersampled)
x_test_lemma_undersampled=vectorizer_lemma_undersample.transform(x_test_undersampled)

In [16]:
vectorizer_lemma_stopkept_undersample=CountVectorizer(tokenizer=lambda x : lemma(x,remove_stopwords=False),max_features=1000)
x_train_lemma_stopkept_undersampled=vectorizer_lemma_stopkept_undersample.fit_transform(x_train_undersampled)
x_test_lemma_stopkept_undersampled=vectorizer_lemma_stopkept_undersample.transform(x_test_undersampled)

### with tf-idf vectorizer

In [13]:
tfidf_stem_undersample=TfidfVectorizer(tokenizer=lambda x : stem(x,remove_stopwords=True),max_features=1000)
x_train_stem_undersampled_tfidf=tfidf_stem_undersample.fit_transform(x_train_undersampled)
x_test_stem_undersampled_tfidf=tfidf_stem_undersample.transform(x_test_undersampled)

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [14]:
tfidf_stem_stopkept_undersample=TfidfVectorizer(tokenizer=lambda x : stem(x,remove_stopwords=False),max_features=1000)
x_train_stem_stopkept_undersampled_tfidf=tfidf_stem_stopkept_undersample.fit_transform(x_train_undersampled)
x_test_stem_stopkept_undersampled_tfidf=tfidf_stem_stopkept_undersample.transform(x_test_undersampled)

In [15]:
tfidf_lemma_undersample=TfidfVectorizer(tokenizer=lambda x : lemma(x,remove_stopwords=True),max_features=1000)
x_train_lemma_undersampled_tfidf=tfidf_lemma_undersample.fit_transform(x_train_undersampled)
x_test_lemma_undersampled_tfidf=tfidf_lemma_undersample.transform(x_test_undersampled)

In [16]:
tfidf_lemma_stopkept_undersample=TfidfVectorizer(tokenizer=lambda x : lemma(x,remove_stopwords=False),max_features=1000)
x_train_lemma_stopkept_undersampled_tfidf=tfidf_lemma_stopkept_undersample.fit_transform(x_train_undersampled)
x_test_lemma_stopkept_undersampled_tfidf=tfidf_lemma_stopkept_undersample.transform(x_test_undersampled)

In [17]:
x_train_lemma_stopkept_undersampled_tfidf.shape

(66774, 1000)

## Models

In [18]:
models_Bow_undersample={
    'SVM': SVC(),
    'MultinomialNB': MultinomialNB(),
    'MLP': MLPClassifier(hidden_layer_sizes=(512,256,128), max_iter=1000,activation='relu',solver='adam',early_stopping=True,alpha=0.0005,batch_size=256,random_state=42)
}

## with countvectorizer

#### stem 

In [24]:
for model_name, model in models_Bow_undersample.items():
    model.fit(x_train_stem_undersampled, y_train_undersampled)
    y_pred = model.predict(x_test_stem_undersampled) #removed
    accuracy=accuracy_score(y_test_undersampled, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test_undersampled, y_pred))
    
    y_pred_train = model.predict(x_train_stem_undersampled) #removed
    accuracy_train=accuracy_score(y_train_undersampled, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    print('-'*50)
    
    results_Bow_undersample['undersample '+model_name+" stem+StopWords removed"] = accuracy

Results for SVM: accuracy=0.6263328141847371
              precision    recall  f1-score   support

           1       0.69      0.71      0.70      2000
           2       0.47      0.73      0.57      2000
           3       0.61      0.72      0.66      2000
           4       0.70      0.66      0.68      1980
           5       0.83      0.78      0.80      1963
           6       0.88      0.60      0.72      1269
           7       0.74      0.60      0.66      1268
           8       0.64      0.44      0.52      1198
           9       0.73      0.50      0.59      1016
          10       0.37      0.37      0.37      2000

    accuracy                           0.63     16694
   macro avg       0.66      0.61      0.63     16694
weighted avg       0.65      0.63      0.63     16694

Results for SVM Training: accuracy=0.765941833647827
--------------------------------------------------
Results for MultinomialNB: accuracy=0.6012938780400143
              precision    recall  f1

#### stem + stopwords kept

In [35]:
for model_name, model in models_Bow_undersample.items():
    model.fit(x_train_stem_stopkept_undersampled, y_train_undersampled)
    y_pred = model.predict(x_test_stem_stopkept_undersampled) 
    accuracy=accuracy_score(y_test_undersampled, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test_undersampled, y_pred))
    
    y_pred_train = model.predict(x_train_stem_stopkept_undersampled) 
    accuracy_train=accuracy_score(y_train_undersampled, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    print('-'*50)
    
    results_Bow_undersample['undersample '+model_name+" stem+StopWords kept"] = accuracy

Results for SVM: accuracy=0.6225590032346952
              precision    recall  f1-score   support

           1       0.68      0.68      0.68      2000
           2       0.47      0.72      0.57      2000
           3       0.60      0.72      0.66      2000
           4       0.70      0.68      0.69      1980
           5       0.83      0.78      0.80      1963
           6       0.90      0.60      0.72      1269
           7       0.75      0.62      0.68      1268
           8       0.62      0.39      0.48      1198
           9       0.74      0.48      0.59      1016
          10       0.35      0.38      0.37      2000

    accuracy                           0.62     16694
   macro avg       0.66      0.61      0.62     16694
weighted avg       0.65      0.62      0.62     16694

Results for SVM Training: accuracy=0.7844071045616557
--------------------------------------------------
Results for MultinomialNB: accuracy=0.5940457649454894
              precision    recall  f

#### lemma

In [38]:
for model_name, model in models_Bow_undersample.items():
    model.fit(x_train_lemma_undersampled, y_train_undersampled)
    y_pred = model.predict(x_test_lemma_undersampled) 
    accuracy=accuracy_score(y_test_undersampled, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test_undersampled, y_pred))
    
    y_pred_train = model.predict(x_train_lemma_undersampled) 
    accuracy_train=accuracy_score(y_train_undersampled, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    print('-'*50)
    
    results_Bow_undersample['undersample '+model_name+" lemma+StopWords removed"] = accuracy

Results for SVM: accuracy=0.6224391997124715
              precision    recall  f1-score   support

           1       0.69      0.71      0.70      2000
           2       0.44      0.73      0.55      2000
           3       0.61      0.71      0.66      2000
           4       0.73      0.65      0.69      1980
           5       0.83      0.78      0.80      1963
           6       0.89      0.60      0.72      1269
           7       0.74      0.60      0.66      1268
           8       0.63      0.40      0.49      1198
           9       0.74      0.50      0.60      1016
          10       0.36      0.36      0.36      2000

    accuracy                           0.62     16694
   macro avg       0.67      0.61      0.62     16694
weighted avg       0.65      0.62      0.63     16694

Results for SVM Training: accuracy=0.7523587024889927
--------------------------------------------------
Results for MultinomialNB: accuracy=0.5948843896010543
              precision    recall  f

#### lemma + stopwords kept

In [40]:
for model_name, model in models_Bow_undersample.items():
    model.fit(x_train_lemma_stopkept_undersampled, y_train_undersampled)
    y_pred = model.predict(x_test_lemma_stopkept_undersampled) 
    accuracy=accuracy_score(y_test_undersampled, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test_undersampled, y_pred))
    
    y_pred_train = model.predict(x_train_lemma_stopkept_undersampled) 
    accuracy_train=accuracy_score(y_train_undersampled, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    print('-'*50)
    
    results_Bow_undersample['undersample '+model_name+" lemma+StopWords kept"] = accuracy

Results for SVM: accuracy=0.6184856834790943
              precision    recall  f1-score   support

           1       0.69      0.68      0.68      2000
           2       0.45      0.73      0.56      2000
           3       0.60      0.73      0.66      2000
           4       0.70      0.68      0.69      1980
           5       0.84      0.77      0.80      1963
           6       0.90      0.59      0.71      1269
           7       0.76      0.60      0.67      1268
           8       0.63      0.38      0.47      1198
           9       0.76      0.48      0.59      1016
          10       0.35      0.38      0.36      2000

    accuracy                           0.62     16694
   macro avg       0.67      0.60      0.62     16694
weighted avg       0.65      0.62      0.62     16694

Results for SVM Training: accuracy=0.7733249468355947
--------------------------------------------------
Results for MultinomialNB: accuracy=0.5880555888343117
              precision    recall  f

In [41]:
results_Bow_undersample

{'undersample SVM stem+StopWords removed': 0.6263328141847371,
 'undersample MultinomialNB stem+StopWords removed': 0.6012938780400143,
 'undersample MLP stem+StopWords removed': 0.6314843656403498,
 'undersample SVM stem+StopWords kept': 0.6225590032346952,
 'undersample MultinomialNB stem+StopWords kept': 0.5940457649454894,
 'undersample MLP stem+StopWords kept': 0.6299868216125554,
 'undersample SVM lemma+StopWords removed': 0.6224391997124715,
 'undersample MultinomialNB lemma+StopWords removed': 0.5948843896010543,
 'undersample MLP lemma+StopWords removed': 0.6245357613513838,
 'undersample SVM lemma+StopWords kept': 0.6184856834790943,
 'undersample MultinomialNB lemma+StopWords kept': 0.5880555888343117,
 'undersample MLP lemma+StopWords kept': 0.6229783155624775}

### HMM

In [43]:
hmm_stem_undersample=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
hmm_stem_stopkept_undersample=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
hmm_lemma_undersample=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
hmm_lemma_stopkept_undersample=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

#### dimension reduction (66774,50)

In [44]:
stem_svd_undersample=TruncatedSVD(n_components=50)
x_train_stem_undersampled_svd=stem_svd_undersample.fit_transform(x_train_stem_undersampled)
x_test_stem_undersampled_svd=stem_svd_undersample.transform(x_test_stem_undersampled)

In [45]:
stem_stopkept_svd_undersample=TruncatedSVD(n_components=50)
x_train_stem_stopkept_undersampled_svd=stem_stopkept_svd_undersample.fit_transform(x_train_stem_stopkept_undersampled)
x_test_stem_stopkept_undersampled_svd=stem_stopkept_svd_undersample.transform(x_test_stem_stopkept_undersampled)

In [46]:
lemma_svd_undersample=TruncatedSVD(n_components=50)
x_train_lemma_undersampled_svd=lemma_svd_undersample.fit_transform(x_train_lemma_undersampled)
x_test_lemma_undersampled_svd=lemma_svd_undersample.transform(x_test_lemma_undersampled)

In [47]:
lemma_stopkept_svd_undersample=TruncatedSVD(n_components=50)
x_train_lemma_stopkept_undersampled_svd=lemma_stopkept_svd_undersample.fit_transform(x_train_lemma_stopkept_undersampled)
x_test_lemma_stopkept_undersampled_svd=lemma_stopkept_svd_undersample.transform(x_test_lemma_stopkept_undersampled)

In [48]:
x_train_stem_undersampled_svd.shape

(66774, 50)

#### apply HMM

In [50]:
lengths_train = [1] * x_train_stem_undersampled_svd.shape[0]
lengths_test = [1] * x_test_stem_undersampled_svd.shape[0]

##### stem

In [51]:
hmm_stem_undersample.fit(x_train_stem_undersampled_svd)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [53]:
y_pred_stem_undersample=hmm_stem_undersample.predict(x_train_stem_undersampled_svd,lengths=lengths_train)
accuracy_hmm_stem_undersample = accuracy_score(y_train_undersampled, y_pred_stem_undersample)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_stem_undersample}")

training HMM BoW Accuracy with lengths: 0.07177943510947375


In [54]:
y_pred_stem_undersample=hmm_stem_undersample.predict(x_test_stem_undersampled_svd,lengths=lengths_test)
accuracy_hmm_stem_undersample = accuracy_score(y_test_undersampled, y_pred_stem_undersample)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_stem_undersample}")

testing HMM BoW Accuracy with lengths: 0.07176230981190847


In [55]:
y_pred_stem_undersample=hmm_stem_undersample.predict(x_test_stem_undersampled_svd)
accuracy_hmm_stem_undersample = accuracy_score(y_test_undersampled, y_pred_stem_undersample)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_stem_undersample}")

testing HMM BoW Accuracy: 0.0770935665508566


In [57]:
results_Bow_undersample['undersample HMM stem+StopWords removed']=0.0770935665508566

##### stem + stopwords kept

In [59]:
hmm_stem_stopkept_undersample.fit(x_train_stem_stopkept_undersampled_svd)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [61]:
y_pred_stem_stopkept_undersample=hmm_stem_stopkept_undersample.predict(x_train_stem_stopkept_undersampled_svd,lengths=lengths_train)
accuracy_hmm_stem_stopkept_undersample = accuracy_score(y_train_undersampled, y_pred_stem_stopkept_undersample)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_stem_stopkept_undersample}")

training HMM BoW Accuracy with lengths: 0.11980711055201126


In [63]:
y_pred_stem_stopkept_undersample=hmm_stem_stopkept_undersample.predict(x_test_stem_stopkept_undersampled_svd,lengths=lengths_test)
accuracy_hmm_stem_stopkept_undersample = accuracy_score(y_test_undersampled, y_pred_stem_stopkept_undersample)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_stem_stopkept_undersample}")

testing HMM BoW Accuracy with lengths: 0.11980352222355337


In [64]:
y_pred_stem_stopkept_undersample=hmm_stem_stopkept_undersample.predict(x_test_stem_stopkept_undersampled_svd)
accuracy_hmm_stem_stopkept_undersample = accuracy_score(y_test_undersampled, y_pred_stem_stopkept_undersample)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_stem_stopkept_undersample}")

testing HMM BoW Accuracy: 0.09230861387324787


In [65]:
results_Bow_undersample['undersample HMM stem+StopWords kept']=0.11980352222355337

##### lemma

In [68]:
hmm_lemma_undersample.fit(x_train_lemma_undersampled_svd)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [69]:
y_pred_lemma_undersample=hmm_lemma_undersample.predict(x_train_lemma_undersampled_svd,lengths=lengths_train)
accuracy_hmm_lemma_undersample = accuracy_score(y_train_undersampled, y_pred_lemma_undersample)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_lemma_undersample}")

training HMM BoW Accuracy with lengths: 0.07177943510947375


In [70]:
y_pred_lemma_undersample=hmm_lemma_undersample.predict(x_test_lemma_undersampled_svd,lengths=lengths_test)
accuracy_hmm_lemma_undersample = accuracy_score(y_test_undersampled, y_pred_lemma_undersample)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_lemma_undersample}")

testing HMM BoW Accuracy with lengths: 0.07176230981190847


In [71]:
y_pred_lemma_undersample=hmm_lemma_undersample.predict(x_test_lemma_undersampled_svd)
accuracy_hmm_lemma_undersample = accuracy_score(y_test_undersampled, y_pred_lemma_undersample)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_lemma_undersample}")

testing HMM BoW Accuracy: 0.08649814304540554


In [83]:
results_Bow_undersample['undersample HMM lemma+StopWords removed']=0.08649814304540554

##### lemma + stopwords kept

In [73]:
hmm_lemma_stopkept_undersample.fit(x_train_lemma_stopkept_undersampled_svd)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [74]:
y_pred_lemma_stopkept_undersample=hmm_lemma_stopkept_undersample.predict(x_train_lemma_stopkept_undersampled_svd,lengths=lengths_train)
accuracy_hmm_lemma_stopkept_undersample = accuracy_score(y_train_undersampled, y_pred_lemma_stopkept_undersample)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_lemma_stopkept_undersample}")

training HMM BoW Accuracy with lengths: 0.0


In [75]:
y_pred_lemma_stopkept_undersample=hmm_lemma_stopkept_undersample.predict(x_test_lemma_stopkept_undersampled_svd,lengths=lengths_test)
accuracy_hmm_lemma_stopkept_undersample = accuracy_score(y_test_undersampled, y_pred_lemma_stopkept_undersample)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_lemma_stopkept_undersample}")

testing HMM BoW Accuracy with lengths: 0.0


In [76]:
y_pred_lemma_stopkept_undersample=hmm_lemma_stopkept_undersample.predict(x_test_lemma_stopkept_undersampled_svd)
accuracy_hmm_lemma_stopkept_undersample = accuracy_score(y_test_undersampled, y_pred_lemma_stopkept_undersample)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_lemma_stopkept_undersample}")

testing HMM BoW Accuracy: 0.098178986462202


In [80]:
results_Bow_undersample['undersample HMM lemma+StopWords kept']=0.098178986462202

#### save the results of HMM

In [84]:
results_Bow_undersample

{'undersample SVM stem+StopWords removed': 0.6263328141847371,
 'undersample MultinomialNB stem+StopWords removed': 0.6012938780400143,
 'undersample MLP stem+StopWords removed': 0.6314843656403498,
 'undersample SVM stem+StopWords kept': 0.6225590032346952,
 'undersample MultinomialNB stem+StopWords kept': 0.5940457649454894,
 'undersample MLP stem+StopWords kept': 0.6299868216125554,
 'undersample SVM lemma+StopWords removed': 0.6224391997124715,
 'undersample MultinomialNB lemma+StopWords removed': 0.5948843896010543,
 'undersample MLP lemma+StopWords removed': 0.6245357613513838,
 'undersample SVM lemma+StopWords kept': 0.6184856834790943,
 'undersample MultinomialNB lemma+StopWords kept': 0.5880555888343117,
 'undersample MLP lemma+StopWords kept': 0.6229783155624775,
 'undersample HMM stem+StopWords removed': 0.0770935665508566,
 'undersample HMM stem+StopWords kept': 0.11980352222355337,
 'undersample HMM lemma+StopWords removed': 0.08649814304540554,
 'undersample HMM lemma+Sto

In [85]:
joblib.dump(results_Bow_undersample,'results_Bow_undersample')

['results_Bow_undersample']

## with tf-idf

### stem

In [19]:
for model_name, model in models_Bow_undersample.items():
    model.fit(x_train_stem_undersampled_tfidf, y_train_undersampled)
    y_pred = model.predict(x_test_stem_undersampled_tfidf) #removed
    accuracy=accuracy_score(y_test_undersampled, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test_undersampled, y_pred))
    
    y_pred_train = model.predict(x_train_stem_undersampled_tfidf) #removed
    accuracy_train=accuracy_score(y_train_undersampled, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    print('-'*50)
    
    results_Bow_undersample['(tf-idf)undersample '+model_name+" stem+StopWords removed"] = accuracy

Results for SVM: accuracy=0.6383131664070923
              precision    recall  f1-score   support

           1       0.69      0.72      0.70      2000
           2       0.51      0.71      0.59      2000
           3       0.61      0.72      0.66      2000
           4       0.69      0.69      0.69      1980
           5       0.83      0.79      0.81      1963
           6       0.89      0.61      0.72      1269
           7       0.73      0.64      0.68      1268
           8       0.61      0.46      0.52      1198
           9       0.72      0.52      0.60      1016
          10       0.39      0.40      0.39      2000

    accuracy                           0.64     16694
   macro avg       0.67      0.62      0.64     16694
weighted avg       0.65      0.64      0.64     16694

Results for SVM Training: accuracy=0.7903525324227993
--------------------------------------------------
Results for MultinomialNB: accuracy=0.5993171199233257
              precision    recall  f

### stem + stopwords kept

In [20]:
for model_name, model in models_Bow_undersample.items():
    model.fit(x_train_stem_stopkept_undersampled_tfidf, y_train_undersampled)
    y_pred = model.predict(x_test_stem_stopkept_undersampled_tfidf) #kept
    accuracy=accuracy_score(y_test_undersampled, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test_undersampled, y_pred))
    
    y_pred_train = model.predict(x_train_stem_stopkept_undersampled_tfidf) #kept
    accuracy_train=accuracy_score(y_train_undersampled, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    print('-'*50)
    
    results_Bow_undersample['(tf-idf)undersample '+model_name+" stem+StopWords kept"] = accuracy

Results for SVM: accuracy=0.6371151311848569
              precision    recall  f1-score   support

           1       0.67      0.71      0.69      2000
           2       0.52      0.69      0.59      2000
           3       0.61      0.73      0.67      2000
           4       0.71      0.71      0.71      1980
           5       0.82      0.79      0.80      1963
           6       0.90      0.61      0.72      1269
           7       0.72      0.67      0.69      1268
           8       0.60      0.43      0.51      1198
           9       0.72      0.51      0.60      1016
          10       0.38      0.39      0.38      2000

    accuracy                           0.64     16694
   macro avg       0.66      0.62      0.64     16694
weighted avg       0.65      0.64      0.64     16694

Results for SVM Training: accuracy=0.8133554976487855
--------------------------------------------------
Results for MultinomialNB: accuracy=0.593626452617707
              precision    recall  f1

### lemma

In [24]:
for model_name, model in models_Bow_undersample.items():
    model.fit(x_train_lemma_undersampled_tfidf, y_train_undersampled)
    y_pred = model.predict(x_test_lemma_undersampled_tfidf) #removed
    accuracy=accuracy_score(y_test_undersampled, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test_undersampled, y_pred))
    
    y_pred_train = model.predict(x_train_lemma_undersampled_tfidf) #removed
    accuracy_train=accuracy_score(y_train_undersampled, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    print('-'*50)
    
    results_Bow_undersample['(tf-idf)undersample '+model_name+" lemma+StopWords removed"] = accuracy

Results for SVM: accuracy=0.6307655445070085
              precision    recall  f1-score   support

           1       0.68      0.72      0.70      2000
           2       0.48      0.70      0.57      2000
           3       0.62      0.71      0.66      2000
           4       0.70      0.69      0.69      1980
           5       0.83      0.78      0.81      1963
           6       0.89      0.61      0.72      1269
           7       0.73      0.63      0.67      1268
           8       0.60      0.42      0.50      1198
           9       0.71      0.53      0.61      1016
          10       0.38      0.38      0.38      2000

    accuracy                           0.63     16694
   macro avg       0.66      0.62      0.63     16694
weighted avg       0.65      0.63      0.63     16694

Results for SVM Training: accuracy=0.7760355827118339
--------------------------------------------------
Results for MultinomialNB: accuracy=0.5944051755121601
              precision    recall  f

### lemma + stopwords kept

In [27]:
for model_name, model in models_Bow_undersample.items():
    model.fit(x_train_lemma_stopkept_undersampled_tfidf, y_train_undersampled)
    y_pred = model.predict(x_test_lemma_stopkept_undersampled_tfidf) #kept
    accuracy=accuracy_score(y_test_undersampled, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test_undersampled, y_pred))
    
    y_pred_train = model.predict(x_train_lemma_stopkept_undersampled_tfidf) #kept
    accuracy_train=accuracy_score(y_train_undersampled, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    print('-'*50)
    
    results_Bow_undersample['(tf-idf)undersample '+model_name+" lemma+StopWords kept"] = accuracy

Results for SVM: accuracy=0.6337606325625973
              precision    recall  f1-score   support

           1       0.68      0.70      0.69      2000
           2       0.50      0.71      0.59      2000
           3       0.61      0.73      0.67      2000
           4       0.70      0.71      0.71      1980
           5       0.84      0.78      0.81      1963
           6       0.89      0.61      0.72      1269
           7       0.74      0.63      0.68      1268
           8       0.60      0.41      0.49      1198
           9       0.73      0.51      0.60      1016
          10       0.37      0.39      0.38      2000

    accuracy                           0.63     16694
   macro avg       0.67      0.62      0.63     16694
weighted avg       0.65      0.63      0.64     16694

Results for SVM Training: accuracy=0.8020037739239824
--------------------------------------------------
Results for MultinomialNB: accuracy=0.5894333293398826
              precision    recall  f

### HMM

In [30]:
hmm_stem_undersample_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
hmm_stem_stopkept_undersample_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
hmm_lemma_undersample_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
hmm_lemma_stopkept_undersample_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

#### dimension reduction (66774,50)

In [31]:
stem_svd_undersample_tfidf=TruncatedSVD(n_components=50)
x_train_stem_undersampled_tfidf_svd=stem_svd_undersample_tfidf.fit_transform(x_train_stem_undersampled_tfidf)
x_test_stem_undersampled_tfidf_svd=stem_svd_undersample_tfidf.transform(x_test_stem_undersampled_tfidf)

In [32]:
stem_stopkept_svd_undersample_tfidf=TruncatedSVD(n_components=50)
x_train_stem_stopkept_undersampled_tfidf_svd=stem_stopkept_svd_undersample_tfidf.fit_transform(x_train_stem_stopkept_undersampled_tfidf)
x_test_stem_stopkept_undersampled_tfidf_svd=stem_stopkept_svd_undersample_tfidf.transform(x_test_stem_stopkept_undersampled_tfidf)

In [33]:
lemma_svd_undersample_tfidf=TruncatedSVD(n_components=50)
x_train_lemma_undersampled_tfidf_svd=lemma_svd_undersample_tfidf.fit_transform(x_train_lemma_undersampled_tfidf)
x_test_lemma_undersampled_tfidf_svd=lemma_svd_undersample_tfidf.transform(x_test_lemma_undersampled_tfidf)

In [34]:
lemma_stopkept_svd_undersample_tfidf=TruncatedSVD(n_components=50)
x_train_lemma_stopkept_undersampled_tfidf_svd=lemma_stopkept_svd_undersample_tfidf.fit_transform(x_train_lemma_stopkept_undersampled_tfidf)
x_test_lemma_stopkept_undersampled_tfidf_svd=lemma_stopkept_svd_undersample_tfidf.transform(x_test_lemma_stopkept_undersampled_tfidf)

#### apply HMM

In [35]:
lengths_train = [1] * x_train_stem_stopkept_undersampled_tfidf_svd.shape[0]
lengths_test = [1] * x_test_stem_stopkept_undersampled_tfidf_svd.shape[0]

In [37]:
lengths_train

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,


##### stem

In [36]:
hmm_stem_undersample_tfidf.fit(x_train_stem_undersampled_tfidf_svd)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [41]:
y_pred_stem_tfidf_undersample=hmm_stem_undersample_tfidf.predict(x_train_stem_undersampled_tfidf_svd,)
accuracy_hmm_stem_tfidf_undersample = accuracy_score(y_train_undersampled, y_pred_stem_tfidf_undersample)   
print(f"training HMM BoW Accuracy : {accuracy_hmm_stem_tfidf_undersample}")

training HMM BoW Accuracy : 0.08810315392218528


In [39]:
y_pred_stem_tfidf_undersample=hmm_stem_undersample_tfidf.predict(x_test_stem_undersampled_tfidf_svd,lengths=lengths_test)
accuracy_hmm_stem_tfidf_undersample = accuracy_score(y_test_undersampled, y_pred_stem_tfidf_undersample)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_stem_tfidf_undersample}")

testing HMM BoW Accuracy with lengths: 0.06086018928956511


In [40]:
y_pred_stem_tfidf_undersample=hmm_stem_undersample_tfidf.predict(x_test_stem_undersampled_tfidf_svd)
accuracy_hmm_stem_tfidf_undersample = accuracy_score(y_test_undersampled, y_pred_stem_tfidf_undersample)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_stem_tfidf_undersample}")

testing HMM BoW Accuracy: 0.09129028393434767


In [48]:
results_Bow_undersample['(tf-idf)undersample HMM stem+StopWords removed']=0.09129028393434767

##### stem + stopwords kept

In [42]:
hmm_stem_stopkept_undersample_tfidf.fit(x_train_stem_stopkept_undersampled_tfidf_svd)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [43]:
y_pred_stem_stopkept_tfidf_undersample=hmm_stem_stopkept_undersample_tfidf.predict(x_train_stem_stopkept_undersampled_tfidf_svd,lengths=lengths_train)
accuracy_hmm_stem_stopkept_tfidf_undersample = accuracy_score(y_train_undersampled, y_pred_stem_stopkept_tfidf_undersample)   
print(f"training HMM BoW Accuracy : {accuracy_hmm_stem_stopkept_tfidf_undersample}")

training HMM BoW Accuracy : 0.11762063078443706


In [44]:
y_pred_stem_stopkept_tfidf_undersample=hmm_stem_stopkept_undersample_tfidf.predict(x_test_stem_stopkept_undersampled_tfidf_svd,lengths=lengths_test)
accuracy_hmm_stem_stopkept_tfidf_undersample = accuracy_score(y_test_undersampled, y_pred_stem_stopkept_tfidf_undersample)   
print(f"testing HMM BoW Accuracy : {accuracy_hmm_stem_stopkept_tfidf_undersample}")

testing HMM BoW Accuracy : 0.11758715706241764


In [45]:
y_pred_stem_stopkept_tfidf_undersample=hmm_stem_stopkept_undersample_tfidf.predict(x_test_stem_stopkept_undersampled_tfidf_svd,)
accuracy_hmm_stem_stopkept_tfidf_undersample = accuracy_score(y_test_undersampled, y_pred_stem_stopkept_tfidf_undersample)   
print(f"testing HMM BoW Accuracy : {accuracy_hmm_stem_stopkept_tfidf_undersample}")

testing HMM BoW Accuracy : 0.0938061579010423


In [46]:
results_Bow_undersample['(tf-idf)undersample HMM stem+StopWords kept']=0.11758715706241764

##### lemma

In [50]:
hmm_lemma_undersample_tfidf.fit(x_train_lemma_undersampled_tfidf_svd)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [51]:
y_pred_lemma_tfidf_undersample=hmm_lemma_undersample_tfidf.predict(x_train_lemma_undersampled_tfidf_svd,lengths=lengths_train)
accuracy_hmm_lemma_tfidf_undersample = accuracy_score(y_train_undersampled, y_pred_lemma_tfidf_undersample)   
print(f"training HMM BoW Accuracy : {accuracy_hmm_lemma_tfidf_undersample}")

training HMM BoW Accuracy : 0.11757570311798005


In [52]:
y_pred_lemma_tfidf_undersample=hmm_lemma_undersample_tfidf.predict(x_test_lemma_undersampled_tfidf_svd,lengths=lengths_test)
accuracy_hmm_lemma_tfidf_undersample = accuracy_score(y_test_undersampled, y_pred_lemma_tfidf_undersample)   
print(f"testing HMM BoW Accuracy with length: {accuracy_hmm_lemma_tfidf_undersample}")

testing HMM BoW Accuracy with length: 0.11758715706241764


In [53]:
y_pred_lemma_tfidf_undersample=hmm_lemma_undersample_tfidf.predict(x_test_lemma_undersampled_tfidf_svd,)
accuracy_hmm_lemma_tfidf_undersample = accuracy_score(y_test_undersampled, y_pred_lemma_tfidf_undersample)   
print(f"testing HMM BoW Accuracy : {accuracy_hmm_lemma_tfidf_undersample}")

testing HMM BoW Accuracy : 0.07391877321193244


In [54]:
results_Bow_undersample['(tf-idf)undersample HMM lemma+StopWords removed']=0.11758715706241764

##### lemma + stopwords kept

In [56]:
hmm_lemma_stopkept_undersample_tfidf.fit(x_train_lemma_stopkept_undersampled_tfidf_svd)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [57]:
y_pred_lemma_stopkept_tfidf_undersample=hmm_lemma_stopkept_undersample_tfidf.predict(x_train_lemma_stopkept_undersampled_tfidf_svd,lengths=lengths_train)
accuracy_hmm_lemma_stopkept_tfidf_undersample = accuracy_score(y_train_undersampled, y_pred_lemma_stopkept_tfidf_undersample)   
print(f"training HMM BoW Accuracy with length : {accuracy_hmm_lemma_stopkept_tfidf_undersample}")

training HMM BoW Accuracy with length : 0.11980711055201126


In [58]:
y_pred_lemma_stopkept_tfidf_undersample=hmm_lemma_stopkept_undersample_tfidf.predict(x_test_lemma_stopkept_undersampled_tfidf_svd,lengths=lengths_test)
accuracy_hmm_lemma_stopkept_tfidf_undersample = accuracy_score(y_test_undersampled, y_pred_lemma_stopkept_tfidf_undersample)   
print(f"testing HMM BoW Accuracy with length : {accuracy_hmm_lemma_stopkept_tfidf_undersample}")

testing HMM BoW Accuracy with length : 0.11980352222355337


In [59]:
y_pred_lemma_stopkept_tfidf_undersample=hmm_lemma_stopkept_undersample_tfidf.predict(x_test_lemma_stopkept_undersampled_tfidf_svd)
accuracy_hmm_lemma_stopkept_tfidf_undersample = accuracy_score(y_test_undersampled, y_pred_lemma_stopkept_tfidf_undersample)   
print(f"testing HMM BoW Accuracy : {accuracy_hmm_lemma_stopkept_tfidf_undersample}")

testing HMM BoW Accuracy : 0.08506050077872289


In [67]:
results_Bow_undersample['(tf-idf)undersample HMM lemma+StopWords keptd']=0.11980352222355337

### save the results

In [61]:
joblib.dump(results_Bow_undersample,'results_Bow_undersample')

['results_Bow_undersample']

In [63]:
len(results_Bow_undersample)

32

In [66]:
with open("results_Bow_undersample.txt", "w", encoding="utf-8") as f:
    for key, value in results_Bow_undersample.items():
        f.write(f"{key}: {value}\n")

Although SVM achieved the highest training accuracy, it suffered from significant overfitting. In contrast, Multinomial Naive Bayes demonstrated strong generalization, while the MLP offered a balanced trade-off between model capacity and generalization.

# ______________________________________________________________________________________

# resampled Data (rosrus)

## BoW feature Extraction

In [10]:
vectorizer_stem_rosrus = CountVectorizer(tokenizer=lambda x: stem(x, remove_stopwords=True))
x_train_stem_rosrus = vectorizer_stem_rosrus.fit_transform(x_train_bal)
x_test_stem_rosrus = vectorizer_stem_rosrus.transform(x_test_bal)

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [11]:
vectorizer_stem_stopKept_rosrus = CountVectorizer(tokenizer=lambda x: stem(x, remove_stopwords=False))
x_train_stem_stopkept_rosrus = vectorizer_stem_stopKept_rosrus.fit_transform(x_train_bal)
x_test_stem_stopkept_rosrus = vectorizer_stem_stopKept_rosrus.transform(x_test_bal)

In [12]:
vectorizer_lemma_rosrus = CountVectorizer(tokenizer=lambda x:lemma(x, remove_stopwords=True))
x_train_lemma_rosrus = vectorizer_lemma_rosrus.fit_transform(x_train_bal)
x_test_lemma_rosrus = vectorizer_lemma_rosrus.transform(x_test_bal)

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [13]:
vectorizer_lemma_stopkept_rosrus = CountVectorizer(tokenizer=lambda x: lemma(x, remove_stopwords=False))
x_train_lemma_stopkept_rosrus = vectorizer_lemma_stopkept_rosrus.fit_transform(x_train_bal)
x_test_lemma_stopkept_rosrus = vectorizer_lemma_stopkept_rosrus.transform(x_test_bal)

## Models (SVM & NB)

In [ ]:
# results_Bow_rosrus={}
results_Bow_rosrus

### Stem + stop words kept

In [37]:
for model_name, model in models_Bow.items():
    model.fit(x_train_stem_stopkept_rosrus, y_train_bal)
    y_pred = model.predict(x_test_stem_stopkept_rosrus) #kept
    accuracy=accuracy_score(y_test_bal, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test_bal, y_pred))
    print('-'*50)
    results_Bow_rosrus['rosrus '+model_name+" stem+StopWords kept"] = accuracy

Results for SVC_BoW: accuracy=0.45715308435747526
              precision    recall  f1-score   support

           1       0.64      0.77      0.70      7120
           2       0.32      0.84      0.46      3589
           3       0.36      0.85      0.50      3473
           4       0.46      0.79      0.58      1980
           5       0.53      0.82      0.64      1963
           6       0.47      0.66      0.55      1269
           7       0.52      0.76      0.62      1268
           8       0.26      0.63      0.37      1198
           9       0.53      0.68      0.59      1016
          10       0.97      0.07      0.13     19029

    accuracy                           0.46     41905
   macro avg       0.50      0.69      0.51     41905
weighted avg       0.70      0.46      0.37     41905

--------------------------------------------------
Results for MultinomialNB_BoW: accuracy=0.559026369168357
              precision    recall  f1-score   support

           1       0.67    

### lemma + stopwords kept

In [39]:
for model_name, model in models_Bow.items():    
    model.fit(x_train_lemma_stopkept_rosrus, y_train_bal)
    y_pred = model.predict(x_test_lemma_stopkept_rosrus) #kept
    accuracy=accuracy_score(y_test_bal, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test_bal, y_pred))
    print('-'*50)   
    results_Bow_rosrus['rosrus '+model_name+" Lemma+StopWords kept"] = accuracy

Results for SVC_BoW: accuracy=0.4530246987233027
              precision    recall  f1-score   support

           1       0.64      0.77      0.70      7120
           2       0.31      0.85      0.46      3589
           3       0.35      0.85      0.50      3473
           4       0.46      0.80      0.58      1980
           5       0.54      0.81      0.65      1963
           6       0.47      0.66      0.55      1269
           7       0.52      0.76      0.62      1268
           8       0.27      0.61      0.37      1198
           9       0.55      0.67      0.60      1016
          10       0.97      0.06      0.11     19029

    accuracy                           0.45     41905
   macro avg       0.51      0.68      0.51     41905
weighted avg       0.70      0.45      0.37     41905

--------------------------------------------------
Results for MultinomialNB_BoW: accuracy=0.5595275026846438
              precision    recall  f1-score   support

           1       0.67    

### Stem + remove stop words

In [ ]:
for model_name, model in models_Bow.items():    
    model.fit(x_train_stem_rosrus, y_train_bal)
    y_pred = model.predict(x_test_stem_rosrus) #removed
    accuracy=accuracy_score(y_test_bal, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test_bal, y_pred))
    print('-'*50)   
    results_Bow['rosrus '+model_name+" stem+StopWords removed"] = accuracy

Results for SVC_BoW: accuracy=0.46693711967545637
              precision    recall  f1-score   support

           1       0.64      0.78      0.70      7120
           2       0.33      0.83      0.47      3589
           3       0.37      0.85      0.51      3473
           4       0.48      0.80      0.60      1980
           5       0.52      0.82      0.64      1963
           6       0.46      0.67      0.55      1269
           7       0.51      0.77      0.62      1268
           8       0.26      0.64      0.37      1198
           9       0.53      0.70      0.60      1016
          10       0.96      0.08      0.15     19029

    accuracy                           0.47     41905
   macro avg       0.51      0.70      0.52     41905
weighted avg       0.70      0.47      0.39     41905

--------------------------------------------------
Results for MultinomialNB_BoW: accuracy=0.5601479537048085
              precision    recall  f1-score   support

           1       0.67   

### Lemma + remove stop words

In [41]:
for model_name, model in models_Bow.items():    
    model.fit(x_train_lemma_rosrus, y_train_bal)
    y_pred = model.predict(x_test_lemma_rosrus) #removed
    accuracy=accuracy_score(y_test_bal, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test_bal, y_pred))
    print('-'*50)   
    results_Bow['rosrus '+model_name+" lemma+StopWords removed"] = accuracy

Results for SVC_BoW: accuracy=0.46404963608161315
              precision    recall  f1-score   support

           1       0.64      0.78      0.70      7120
           2       0.32      0.85      0.46      3589
           3       0.36      0.85      0.51      3473
           4       0.48      0.80      0.60      1980
           5       0.53      0.82      0.65      1963
           6       0.46      0.67      0.55      1269
           7       0.52      0.77      0.62      1268
           8       0.27      0.63      0.38      1198
           9       0.55      0.69      0.61      1016
          10       0.97      0.07      0.14     19029

    accuracy                           0.46     41905
   macro avg       0.51      0.69      0.52     41905
weighted avg       0.70      0.46      0.38     41905

--------------------------------------------------
Results for MultinomialNB_BoW: accuracy=0.5621763512707314
              precision    recall  f1-score   support

           1       0.67   

## HMM  (generative unsupervised)

hmm require dense data so use .toarray() is a solution but the shape (130000, 44211) is too large to allocate in memory

using dimension reduction : PCA also requires dense data(.toarray)--> same problem

In [29]:
hmm_BOW = GaussianHMM(n_components=num_classes, covariance_type="diag", n_iter=100, random_state=42)

In [31]:
hmm_BOW.fit(x_train_lemma_rosrus.toarray())

MemoryError: Unable to allocate 42.8 GiB for an array with shape (130000, 44211) and data type float64

### dimension reduction

TruncatedSVD :faster and accept sparse and dense data

In [15]:
svd_stem_stopkept_rosrus=TruncatedSVD(n_components=50)
x_train_bow_svd_stem_stopkept_rosrus=svd_stem_stopkept_rosrus.fit_transform(x_train_stem_stopkept_rosrus)
x_test_bow_svd_stem_stopkept_rosrus=svd_stem_stopkept_rosrus.transform(x_test_stem_stopkept_rosrus)

In [16]:
svd_stem_rosrus=TruncatedSVD(n_components=50)
x_train_bow_svd_stem_rosrus=svd_stem_rosrus.fit_transform(x_train_stem_rosrus)
x_test_bow_svd_stem_rosrus=svd_stem_rosrus.transform(x_test_stem_rosrus)

In [17]:
svd_lemma_stopkept_rosrus=TruncatedSVD(n_components=50)
x_train_bow_svd_lemma_stopkept_rosrus=svd_lemma_stopkept_rosrus.fit_transform(x_train_lemma_stopkept_rosrus)
x_test_bow_svd_lemma_stopkept_rosrus=svd_lemma_stopkept_rosrus.transform(x_test_lemma_stopkept_rosrus)

In [18]:
svd_lemma_rosrus=TruncatedSVD(n_components=50)
x_train_bow_svd_lemma_rosrus=svd_lemma_rosrus.fit_transform(x_train_lemma_rosrus)
x_test_bow_svd_lemma_rosrus=svd_lemma_rosrus.transform(x_test_lemma_rosrus)

In [48]:
hmm_stem_bow_rosrus=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
hmm_stem_stopkept_bow_rosrus=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
hmm_lemma_bow_rosrus=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
hmm_lemma_stopkept_bow_rosrus=GaussianHMM(n_components=num_classes, algorithm='viterbi',  n_iter=100, random_state=42)

In [45]:
lengths_train = [1] * x_train_bow_svd_stem_rosrus.shape[0]
lengths_test = [1] * x_test_bow_svd_stem_rosrus.shape[0]

### stem + remove stopwords

In [49]:
hmm_stem_bow_rosrus.fit(x_train_bow_svd_stem_rosrus)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [66]:
y_pred_stem_bow_rosrus=hmm_stem_bow_rosrus.predict(x_train_bow_svd_stem_rosrus,lengths=lengths_train)
accuracy_hmm_BOW_stem_rosrus = accuracy_score(y_train_bal, y_pred_stem_bow_rosrus)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_BOW_stem_rosrus}")

training HMM BoW Accuracy with lengths: 0.1


In [101]:
y_pred_stem_bow_rosrus=hmm_stem_bow_rosrus.predict(x_test_bow_svd_stem_rosrus,lengths=lengths_test)
accuracy_hmm_BOW_stem_rosrus = accuracy_score(y_test_bal, y_pred_stem_bow_rosrus)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_BOW_stem_rosrus}")

testing HMM BoW Accuracy with lengths: 0.1699081255220141


In [52]:
y_pred_stem_bow_rosrus=hmm_stem_bow_rosrus.predict(x_test_bow_svd_stem_rosrus)
accuracy_hmm_BOW_stem_rosrus = accuracy_score(y_test_bal, y_pred_stem_bow_rosrus)   
print(f"HMM BoW Accuracy: {accuracy_hmm_BOW_stem_rosrus}")

HMM BoW Accuracy: 0.054026965755876385


In [92]:
results_Bow_rosrus['rosrus HMM_BoW stem+StopWords removed']=0.1699081255220141

### Stem + stopwords kept

In [58]:
hmm_stem_stopkept_bow_rosrus.fit(x_train_bow_svd_stem_stopkept_rosrus)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [74]:
y_pred_stem_stopkept_bow_rosrus=hmm_stem_stopkept_bow_rosrus.predict(x_train_bow_svd_stem_stopkept_rosrus,lengths=lengths_train)
accuracy_hmm_BOW_stem_stopkept_rosrus = accuracy_score(y_train_bal, y_pred_stem_stopkept_bow_rosrus)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_BOW_stem_stopkept_rosrus}")

training HMM BoW Accuracy with lengths: 0.1


In [75]:
y_pred_stem_stopkept_bow_rosrus=hmm_stem_stopkept_bow_rosrus.predict(x_test_bow_svd_stem_stopkept_rosrus,lengths=lengths_test)
accuracy_hmm_BOW_stem_stopkept_rosrus = accuracy_score(y_test_bal, y_pred_stem_stopkept_bow_rosrus)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_BOW_stem_stopkept_rosrus}")

testing HMM BoW Accuracy with lengths: 0.082877938193533


In [76]:
y_pred_stem_stopkept_bow_rosrus=hmm_stem_stopkept_bow_rosrus.predict(x_test_bow_svd_stem_stopkept_rosrus)
accuracy_hmm_BOW_stem_stopkept_rosrus = accuracy_score(y_test_bal, y_pred_stem_stopkept_bow_rosrus)   
print(f"HMM BoW Accuracy with lengths: {accuracy_hmm_BOW_stem_stopkept_rosrus}")

HMM BoW Accuracy with lengths: 0.05901443741796922


In [93]:
results_Bow_rosrus['rosrus HMM_BoW stem+StopWords kept']=0.082877938193533

In [78]:
results_Bow_rosrus

{'rosrus SVC_BoW stem+StopWords kept': 0.45715308435747526,
 'rosrus MultinomialNB_BoW stem+StopWords kept': 0.559026369168357,
 'rosrus SVC_BoW Lemma+StopWords kept': 0.4530246987233027,
 'rosrus MultinomialNB_BoW Lemma+StopWords kept': 0.5595275026846438,
 'rosrus HMM_BoW stem+StopWords removed': 0.1699081255220141,
 'rosrus HMM_BoW stem+StopWords kept': 0.082877938193533}

### lemma + remove stopwords

In [71]:
hmm_lemma_bow_rosrus.fit(x_train_bow_svd_lemma_rosrus)

Model is not converging.  Current: 9298625.125124881 is not greater than 9298625.801044906. Delta is -0.6759200245141983


GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [72]:
y_pred_lemma_bow_rosrus=hmm_lemma_bow_rosrus.predict(x_train_bow_svd_lemma_rosrus,lengths=lengths_train)
accuracy_hmm_BOW_lemma_rosrus = accuracy_score(y_train_bal, y_pred_lemma_bow_rosrus)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_BOW_lemma_rosrus}")

training HMM BoW Accuracy with lengths: 0.1


In [73]:
y_pred_lemma_bow_rosrus=hmm_lemma_bow_rosrus.predict(x_test_bow_svd_lemma_rosrus,lengths=lengths_test)
accuracy_hmm_BOW_lemma_rosrus = accuracy_score(y_test_bal, y_pred_lemma_bow_rosrus)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_BOW_lemma_rosrus}")

testing HMM BoW Accuracy with lengths: 0.024245316787972794


In [81]:
y_pred_lemma_bow_rosrus=hmm_lemma_bow_rosrus.predict(x_test_bow_svd_lemma_rosrus)
accuracy_hmm_BOW_lemma_rosrus = accuracy_score(y_test_bal, y_pred_lemma_bow_rosrus)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_BOW_lemma_rosrus}")

testing HMM BoW Accuracy: 0.060350793461400785


In [ ]:
results_Bow_rosrus

{'rosrus SVC_BoW stem+StopWords kept': 0.45715308435747526,
 'rosrus MultinomialNB_BoW stem+StopWords kept': 0.559026369168357,
 'rosrus SVC_BoW Lemma+StopWords kept': 0.4530246987233027,
 'rosrus MultinomialNB_BoW Lemma+StopWords kept': 0.5595275026846438,
 'rosrus HMM_BoW stem+StopWords removed': 0.1699081255220141,
 'rosrus HMM_BoW stem+StopWords kept': 0.082877938193533}

In [97]:
results_Bow_rosrus['rosrus HMM_BoW lemma+StopWords removed']=0.060350793461400785

### lemma + stop words kept

In [83]:
hmm_lemma_stopkept_bow_rosrus.fit(x_train_bow_svd_lemma_stopkept_rosrus)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [84]:
y_pred_lemma_stopkept_bow_rosrus=hmm_lemma_bow_rosrus.predict(x_train_bow_svd_lemma_stopkept_rosrus,lengths=lengths_train)
accuracy_hmm_BOW_lemma_stopkept_rosrus = accuracy_score(y_train_bal, y_pred_lemma_stopkept_bow_rosrus)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_BOW_lemma_stopkept_rosrus}")

training HMM BoW Accuracy with lengths: 0.1


In [85]:
y_pred_lemma_stopkept_bow_rosrus=hmm_lemma_bow_rosrus.predict(x_test_bow_svd_lemma_stopkept_rosrus,lengths=lengths_test)
accuracy_hmm_BOW_lemma_stopkept_rosrus = accuracy_score(y_test_bal, y_pred_lemma_stopkept_bow_rosrus)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_BOW_lemma_stopkept_rosrus}")

testing HMM BoW Accuracy with lengths: 0.024245316787972794


In [86]:
y_pred_lemma_stopkept_bow_rosrus=hmm_lemma_bow_rosrus.predict(x_test_bow_svd_lemma_stopkept_rosrus)
accuracy_hmm_BOW_lemma_stopkept_rosrus = accuracy_score(y_test_bal, y_pred_lemma_stopkept_bow_rosrus)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_BOW_lemma_stopkept_rosrus}")

testing HMM BoW Accuracy: 0.022097601718172055


In [95]:
results_Bow_rosrus['rosrus HMM_BoW lemma+StopWords kept']=0.024245316787972794

## MLP with dimension reduction (130000, 50)

In [110]:
x_train_stem_rosrus.shape

(130000, 36766)

In [111]:
x_train_bow_svd_stem_rosrus.shape

(130000, 50)

### scaling

In [115]:
scaler_stem=StandardScaler()
scaler_stem_stopkept=StandardScaler()
scaler_lemma=StandardScaler()
scaler_lemma_stopkept=StandardScaler()

In [116]:
x_train_svd_stem_rosrus_scaled=scaler_stem.fit_transform(x_train_bow_svd_stem_rosrus)
x_test_svd_stem_rosrus_scaled=scaler_stem.transform(x_test_bow_svd_stem_rosrus)

In [117]:
x_train_svd_stem_stopkept_rosrus_scaled=scaler_stem_stopkept.fit_transform(x_train_bow_svd_stem_stopkept_rosrus)
x_test_svd_stem_stopkept_rosrus_scaled=scaler_stem_stopkept.transform(x_test_bow_svd_stem_stopkept_rosrus)

In [118]:
x_train_svd_lemma_rosrus_scaled=scaler_lemma.fit_transform(x_train_bow_svd_lemma_rosrus)
x_test_svd_lemma_rosrus_scaled=scaler_lemma.transform(x_test_bow_svd_lemma_rosrus)

In [119]:
x_train_svd_lemma_stopkept_rosrus_scaled=scaler_lemma_stopkept.fit_transform(x_train_bow_svd_lemma_stopkept_rosrus)
x_test_svd_lemma_stopkept_rosrus_scaled=scaler_lemma_stopkept.transform(x_test_bow_svd_lemma_stopkept_rosrus)

In [126]:
stem_svd_MLP=MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000,activation='relu',solver='adam',alpha=0.001)
stem_stopkept_svd_MLP=MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000,activation='relu',solver='adam',alpha=0.001)
lemma_svd_MLP=MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000,activation='relu',solver='adam',alpha=0.001)
lemma_stopkept_svd_MLP=MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000,activation='relu',solver='adam',alpha=0.001)

### stem

In [127]:
stem_svd_MLP.fit(x_train_svd_stem_rosrus_scaled,y_train_bal)

y_pred_stem_mlp_train=stem_svd_MLP.predict(x_train_svd_stem_rosrus_scaled)
print(accuracy_score(y_pred_stem_mlp_train, y_train_bal))

y_pred_stem_mlp=stem_svd_MLP.predict(x_test_svd_stem_rosrus_scaled)
print(accuracy_score(y_pred_stem_mlp, y_test_bal))

0.6299076923076923
0.3688342679871137


In [130]:
results_Bow_rosrus['rosrus MLP stem+stopwords removed']=accuracy_score(y_pred_stem_mlp, y_test_bal)
results_Bow_rosrus

{'rosrus SVC_BoW stem+StopWords kept': 0.45715308435747526,
 'rosrus MultinomialNB_BoW stem+StopWords kept': 0.559026369168357,
 'rosrus SVC_BoW Lemma+StopWords kept': 0.4530246987233027,
 'rosrus MultinomialNB_BoW Lemma+StopWords kept': 0.5595275026846438,
 'rosrus HMM_BoW stem+StopWords removed': 0.1699081255220141,
 'rosrus HMM_BoW stem+StopWords kept': 0.082877938193533,
 'rosrus HMM_BoW lemma+StopWords kept': 0.024245316787972794,
 'rosrus HMM_BoW lemma+StopWords removed': 0.060350793461400785,
 'rosrus MLP stem+stopwords removed': 0.3688342679871137}

### stem + stopwords kept

In [131]:
stem_stopkept_svd_MLP.fit(x_train_svd_stem_stopkept_rosrus_scaled,y_train_bal)

y_pred_stem_stopkept_mlp_train=stem_svd_MLP.predict(x_train_svd_stem_stopkept_rosrus_scaled)
print(accuracy_score(y_pred_stem_stopkept_mlp_train, y_train_bal))

y_pred_stem_stopkept_mlp=stem_svd_MLP.predict(x_test_svd_stem_stopkept_rosrus_scaled)
print(accuracy_score(y_pred_stem_stopkept_mlp, y_test_bal))

0.10434615384615385
0.07729387901205106


In [132]:
results_Bow_rosrus['rosrus MLP stem+stopwords kept']=accuracy_score(y_pred_stem_stopkept_mlp, y_test_bal)

### lemma

In [152]:
lemma_svd_MLP.fit(x_train_svd_lemma_rosrus_scaled,y_train_bal)

y_pred_lemma_mlp_train=stem_svd_MLP.predict(x_train_svd_lemma_rosrus_scaled)
print(accuracy_score(y_pred_lemma_mlp_train, y_train_bal))

y_pred_lemma_mlp=stem_svd_MLP.predict(x_test_svd_lemma_rosrus_scaled)
print(accuracy_score(y_pred_lemma_mlp, y_test_bal))

0.19458461538461538
0.12979358071829136


In [153]:
results_Bow_rosrus['rosrus MLP lemma+stopwords removed']=accuracy_score(y_pred_lemma_mlp, y_test_bal)

### lemma + stopwords kept

In [135]:
lemma_stopkept_svd_MLP.fit(x_train_svd_lemma_stopkept_rosrus_scaled,y_train_bal)

y_pred_lemma_stopkept_mlp_train=stem_svd_MLP.predict(x_train_svd_lemma_stopkept_rosrus_scaled)
print(accuracy_score(y_pred_lemma_stopkept_mlp_train, y_train_bal))

y_pred_lemma_stopkept_mlp=stem_svd_MLP.predict(x_test_svd_lemma_stopkept_rosrus_scaled)
print(accuracy_score(y_pred_lemma_stopkept_mlp, y_test_bal))

0.11286923076923076
0.07932227657797399


In [136]:
results_Bow_rosrus['rosrus MLP lemma+stopwords kept']=accuracy_score(y_pred_lemma_stopkept_mlp, y_test_bal)

In [137]:
results_Bow_rosrus

{'rosrus SVC_BoW stem+StopWords kept': 0.45715308435747526,
 'rosrus MultinomialNB_BoW stem+StopWords kept': 0.559026369168357,
 'rosrus SVC_BoW Lemma+StopWords kept': 0.4530246987233027,
 'rosrus MultinomialNB_BoW Lemma+StopWords kept': 0.5595275026846438,
 'rosrus HMM_BoW stem+StopWords removed': 0.1699081255220141,
 'rosrus HMM_BoW stem+StopWords kept': 0.082877938193533,
 'rosrus HMM_BoW lemma+StopWords kept': 0.024245316787972794,
 'rosrus HMM_BoW lemma+StopWords removed': 0.060350793461400785,
 'rosrus MLP stem+stopwords removed': 0.3688342679871137,
 'rosrus MLP stem+stopwords kept': 0.07729387901205106,
 'rosrus MLP lemma+stopwords removed': 0.12979358071829136,
 'rosrus MLP lemma+stopwords kept': 0.07932227657797399}

## Save results

In [160]:
joblib.dump(results_Bow_rosrus,'results_Bow_rosrus')

['results_Bow_rosrus']

In [161]:
results_Bow_rosrus

{'rosrus SVC_BoW stem+StopWords kept': 0.45715308435747526,
 'rosrus MultinomialNB_BoW stem+StopWords kept': 0.559026369168357,
 'rosrus SVC_BoW Lemma+StopWords kept': 0.4530246987233027,
 'rosrus MultinomialNB_BoW Lemma+StopWords kept': 0.5595275026846438,
 'rosrus HMM_BoW stem+StopWords removed': 0.1699081255220141,
 'rosrus HMM_BoW stem+StopWords kept': 0.082877938193533,
 'rosrus HMM_BoW lemma+StopWords kept': 0.024245316787972794,
 'rosrus HMM_BoW lemma+StopWords removed': 0.060350793461400785,
 'rosrus MLP stem+stopwords removed': 0.3688342679871137,
 'rosrus MLP stem+stopwords kept': 0.07729387901205106,
 'rosrus MLP lemma+stopwords removed': 0.12979358071829136,
 'rosrus MLP lemma+stopwords kept': 0.07932227657797399,
 'rosrus SVC_BoW stem+StopWords removed': 0.46693711967545637,
 'rosrus MultinomialNB_BoW stem+StopWords removed': 0.5601479537048085,
 'rosrus SVC_BoW lemma+stopwords removed': 0.46404963608161315,
 'rosrus MultinomialNB_BoW lemma+stopwords removed': 0.5621763512

SVM performed best with stem + stopwords removed  --> 46.7%

NB performed best with lemma + stopwords removed  -->  56.22%

MLP performed best with stem + stopwords removed  --> 36.88%

HMM performed poorly due to the lack of sequential structure in short news headlines

In [242]:
results_Bow_rosrus=joblib.load('results_Bow_rosrus')
results_Bow_rosrus

{'rosrus SVC_BoW stem+StopWords kept': 0.45715308435747526,
 'rosrus MultinomialNB_BoW stem+StopWords kept': 0.559026369168357,
 'rosrus SVC_BoW Lemma+StopWords kept': 0.4530246987233027,
 'rosrus MultinomialNB_BoW Lemma+StopWords kept': 0.5595275026846438,
 'rosrus HMM_BoW stem+StopWords removed': 0.1699081255220141,
 'rosrus HMM_BoW stem+StopWords kept': 0.082877938193533,
 'rosrus HMM_BoW lemma+StopWords kept': 0.024245316787972794,
 'rosrus HMM_BoW lemma+StopWords removed': 0.060350793461400785,
 'rosrus MLP stem+stopwords removed': 0.3688342679871137,
 'rosrus MLP stem+stopwords kept': 0.07729387901205106,
 'rosrus MLP lemma+stopwords removed': 0.12979358071829136,
 'rosrus MLP lemma+stopwords kept': 0.07932227657797399,
 'rosrus SVC_BoW stem+StopWords removed': 0.46693711967545637,
 'rosrus MultinomialNB_BoW stem+StopWords removed': 0.5601479537048085,
 'rosrus SVC_BoW lemma+stopwords removed': 0.46404963608161315,
 'rosrus MultinomialNB_BoW lemma+stopwords removed': 0.5621763512

In [243]:
with open("results_Bow_rosrus.txt", "w", encoding="utf-8") as f:
    for key, value in results_Bow_rosrus.items():
        f.write(f"{key}: {value}\n")